# Glacial Lake Segmentation — ResNet34-FCN Baseline Colab Training

This notebook is the **dedicated Google Colab training and evaluation environment** for the **ResNet-34 Fully Convolutional Network (FCN)** baseline model.

---

### Key Architectural Highlights
* **Encoder**: ResNet-34 backbone with ImageNet-pretrained weights (`ResNet34_Weights.DEFAULT`).
* **Decoder**: Lightweight 5-stage progressive transposed-convolution upsampling head ($512 \to 256 \to 128 \to 64 \to 32 \to 16 \to 1$) with Batch Normalization and ReLU activations.
* **Parameters**: **~24.08M total** (~21.28M encoder + ~2.79M decoder).
* **Role**: Serves as our convolutional baseline for comparison with the advanced **DeepLabV3+** architecture (~40.35M parameters, ASPP + boundary decoder).

---

### Workflow & Storage Isolation
1. **Google Drive Sync**: All checkpoints, logs, plots, and evaluation reports are saved in a separate Drive folder:
   `Glacial_Lake_Segmentation/experiments/resnet34_fcn_colab/`
2. **Local SSD Caching**: Training and validation image patches ($512 \times 512$) are copied once from Google Drive to the fast Colab local NVMe SSD (`/content/dataset/`) to maximize GPU throughput and prevent Drive I/O bottlenecks.
3. **Mixed Precision (AMP)**: Native PyTorch `torch.cuda.amp` accelerates training on Tesla T4, V100, or A100 GPUs.
4. **Resumable Training**: Atomic checkpointing ensures seamless recovery from Colab session disconnections.

> **Hardware Requirement**: Before executing, select **Runtime → Change runtime type → T4 GPU** (or A100/V100).


## 1. Hardware & Environment Verification


In [ ]:
# 1.1 GPU and CUDA Verification
import sys
import torch

print("=" * 60)
print("  Environment & Hardware Check")
print("=" * 60)
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not detected! Please navigate to:"
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU Model         : {gpu_name}")
print(f"GPU VRAM          : {gpu_mem_gb:.2f} GB")
print("=" * 60)

# Enable cuDNN benchmark for faster convolutional operations with fixed image sizes
torch.backends.cudnn.benchmark = True


## 2. Google Drive Mounting & Repository Setup


In [ ]:
# 2.1 Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("✓ Google Drive mounted successfully.")


In [ ]:
# 2.2 Clone or Update the Glacial Lake Segmentation Repository
from pathlib import Path

%cd /content
REPO_DIR = Path("/content/glacial-lake-segmentation")
REPO_URL = "https://github.com/ShubhamSnSharma/glacial-lake-segmentation.git"

if not REPO_DIR.exists():
    print(f"Cloning repository from {REPO_URL}...")
    !git clone -q {REPO_URL}
else:
    print("Repository already exists locally. Pulling latest commits...")
    %cd {REPO_DIR}
    !git pull -q

%cd {REPO_DIR}
!git log -1 --oneline
print("✓ Active working directory:", os.getcwd())


In [ ]:
# 2.3 Install project dependencies without overriding Colab's CUDA PyTorch
!pip install -q pillow pyyaml matplotlib tqdm scikit-learn pandas

# Add repository root to Python path
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("✓ Dependencies verified and repository path registered.")


## 3. Directory Configuration & Dataset Verification
Configure the paths for the dataset in Google Drive, the local SSD cache, and the dedicated ResNet34-FCN experiment output directory.


In [ ]:
# 3.1 Path definitions
from pathlib import Path

# Base Google Drive project directory
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Glacial_Lake_Segmentation")

# Source dataset location in Google Drive
# Supports GLID and val directories directly under DRIVE_PROJECT_ROOT
DRIVE_DATA_ROOT = DRIVE_PROJECT_ROOT

# Dedicated experiment directory for ResNet34-FCN (completely isolated from DeepLabV3+)
EXP_DIR = DRIVE_PROJECT_ROOT / "experiments" / "resnet34_fcn_colab"
EXP_CHECKPOINTS = EXP_DIR / "checkpoints"
EXP_RESULTS = EXP_DIR / "results"
EXP_LOGS = EXP_RESULTS / "logs"
EXP_PLOTS = EXP_RESULTS / "plots"
EXP_PREDICTIONS = EXP_RESULTS / "predictions"
EXP_EVALUATION = EXP_RESULTS / "evaluation"

# Create persistent output directories on Google Drive
for folder in [EXP_CHECKPOINTS, EXP_LOGS, EXP_PLOTS, EXP_PREDICTIONS, EXP_EVALUATION]:
    folder.mkdir(parents=True, exist_ok=True)

# Fast local Colab SSD path
LOCAL_DATA_ROOT = Path("/content/dataset")
LOCAL_TRAIN_DIR = LOCAL_DATA_ROOT / "GLID"
LOCAL_VAL_DIR = LOCAL_DATA_ROOT / "val"

print("✓ Path Configuration:")
print(f"  Drive Dataset   : {DRIVE_DATA_ROOT}")
print(f"  Local SSD Cache : {LOCAL_DATA_ROOT}")
print(f"  FCN Output Dir  : {EXP_DIR}")


In [ ]:
# 3.2 Verify Drive dataset structure and copy to local SSD
import shutil
import time
from src.dataset import discover_split_paths

# Verify source dataset directories in Drive
drive_train_candidates = [
    DRIVE_DATA_ROOT / "GLID",
    DRIVE_DATA_ROOT / "training",
    DRIVE_DATA_ROOT / "train"
]
drive_val_candidates = [
    DRIVE_DATA_ROOT / "val",
    DRIVE_DATA_ROOT / "validation"
]

src_train_dir = next((p for p in drive_train_candidates if p.is_dir()), None)
src_val_dir = next((p for p in drive_val_candidates if p.is_dir()), None)

if src_train_dir is None or src_val_dir is None:
    raise FileNotFoundError(
        f"Could not find GLID training and val directories in {DRIVE_DATA_ROOT}.\n"
        f"Available items: {[p.name for p in DRIVE_DATA_ROOT.iterdir() if p.is_dir()]}\n"
        f"Please update DRIVE_DATA_ROOT to point to the correct folder."
    )

print(f"Found training source   : {src_train_dir}")
print(f"Found validation source : {src_val_dir}")

# Fast one-time copy to local NVMe SSD
if LOCAL_TRAIN_DIR.exists() and LOCAL_VAL_DIR.exists():
    print("✓ Local SSD dataset already cached. Skipping copy.")
else:
    print("\nCopying dataset from Google Drive to local Colab NVMe SSD...")
    t0 = time.time()
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    
    if not LOCAL_TRAIN_DIR.exists():
        print(f"  Copying {src_train_dir.name} -> {LOCAL_TRAIN_DIR} ...")
        shutil.copytree(src_train_dir, LOCAL_TRAIN_DIR)
        
    if not LOCAL_VAL_DIR.exists():
        print(f"  Copying {src_val_dir.name} -> {LOCAL_VAL_DIR} ...")
        shutil.copytree(src_val_dir, LOCAL_VAL_DIR)
        
    elapsed = (time.time() - t0) / 60
    print(f"✓ Dataset copy complete in {elapsed:.1f} minutes.")


In [ ]:
# 3.3 Validate sample discovery and counts on local SSD
from src.dataset import GLIDDataset

train_ds = GLIDDataset(str(LOCAL_TRAIN_DIR), is_train=True, split="train")
val_ds = GLIDDataset(str(LOCAL_VAL_DIR), is_train=False, split="val")

n_train = len(train_ds)
n_val = len(val_ds)

print("=" * 60)
print("  GLID Dataset Discovery Summary")
print("=" * 60)
print(f"Training Samples   : {n_train:,} (Expected: 16,000)")
print(f"Validation Samples : {n_val:,} (Expected: 2,367)")
print("=" * 60)

if n_train != 16000 or n_val != 2367:
    print(f"[Warning] Sample count differs from standard split ({n_train} train, {n_val} val).")
else:
    print("✓ Local dataset perfectly matches official GLID benchmark split.")


## 4. PyTorch DataLoaders & Architecture Setup


In [ ]:
# 4.1 Create DataLoaders with Colab T4-optimized settings
from torch.utils.data import DataLoader

# Training hyperparameters
BATCH_SIZE = 8          # Reduce to 4 if encountering CUDA Out Of Memory (OOM)
NUM_WORKERS = 2         # Multi-process DataLoader for fast GPU feeding
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=False,
)

print(f"✓ DataLoaders initialized:")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Train batches    : {len(train_loader):,}")
print(f"  Val batches      : {len(val_loader):,}")


In [ ]:
# 4.2 Verify tensor shapes and values from a real batch
sample_images, sample_masks = next(iter(train_loader))

print("Batch Verification:")
print(f"  Images tensor shape : {sample_images.shape} | dtype: {sample_images.dtype}")
print(f"  Masks tensor shape  : {sample_masks.shape} | dtype: {sample_masks.dtype}")
print(f"  Unique mask values  : {torch.unique(sample_masks).tolist()}")

assert sample_images.shape == (BATCH_SIZE, 3, 512, 512), "Image shape mismatch!"
assert sample_masks.shape == (BATCH_SIZE, 1, 512, 512), "Mask shape mismatch!"
assert set(torch.unique(sample_masks).tolist()).issubset({0.0, 1.0}), "Mask must be binary {0.0, 1.0}!"
print("✓ Batch shape and contract validation passed.")


In [ ]:
# 4.3 Initialize ResNet34-FCN baseline model
from src.model import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fcn_cfg = {
    "model": {
        "name": "resnet34_fcn",
        "pretrained": True,
        "pretrained_source": "imagenet",
        "num_classes": 1,
        "freeze_encoder": False
    }
}

model = build_model(fcn_cfg).to(device)
model = model.to(memory_format=torch.channels_last)

summary = model.get_parameter_summary()
print("=" * 60)
print("  ResNet34-FCN Baseline Architecture Summary")
print("=" * 60)
print(f"Total Parameters    : {summary['total_parameters']:,}")
print(f"Encoder (ResNet-34) : {summary['encoder_parameters']:,}")
print(f"Decoder (5-Stage)   : {summary['decoder_parameters']:,}")
print("=" * 60)

# Forward pass test with AMP
model.eval()
with torch.no_grad():
    x = sample_images.to(device, memory_format=torch.channels_last)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        out = model(x)

print(f"Forward pass output : {out.shape} (Expected: ({BATCH_SIZE}, 1, 512, 512))")
assert out.shape == (BATCH_SIZE, 1, 512, 512)
print("✓ Model initialization and AMP forward pass successful.")


In [ ]:
# 4.4 GPU training speed benchmark (2 batches)
import time
from src.losses import CombinedBCEDiceLoss

bench_loss = CombinedBCEDiceLoss()
bench_opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
bench_scaler = torch.amp.GradScaler("cuda")

times = []
model.train()

for i, (imgs, msks) in enumerate(train_loader):
    if i >= 2:
        break
    t_start = time.time()
    
    imgs = imgs.to(device, non_blocking=True, memory_format=torch.channels_last)
    msks = msks.to(device, non_blocking=True)
    
    bench_opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        preds = model(imgs)
        loss = bench_loss(preds, msks)
        
    bench_scaler.scale(loss).backward()
    bench_scaler.unscale_(bench_opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    bench_scaler.step(bench_opt)
    bench_scaler.update()
    
    torch.cuda.synchronize()
    times.append(time.time() - t_start)
    print(f"  Benchmark batch {i+1}/2 | Loss: {loss.item():.4f} | Time: {times[-1]:.3f}s")

avg_batch_time = sum(times) / len(times)
est_epoch_min = (avg_batch_time * len(train_loader)) / 60
print(f"\n✓ Benchmark complete: Average batch time = {avg_batch_time:.3f}s")
print(f"  Estimated training time per epoch = {est_epoch_min:.1f} minutes")


## 5. Checkpoint Management & Resume Configuration
Inspect existing checkpoints in your Google Drive experiment directory to decide whether to resume an in-progress run or train from scratch.


In [ ]:
# 5.1 Checkpoint Inspection
from pathlib import Path

LATEST_CHECKPOINT_PATH = EXP_CHECKPOINTS / "latest_model.pth"
BEST_CHECKPOINT_PATH = EXP_CHECKPOINTS / "best_model.pth"

# Training configuration options
TOTAL_EPOCHS = 30
WARMUP_EPOCHS = 3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Toggle whether to resume automatically if latest_model.pth exists
RESUME_IF_AVAILABLE = True

print("=" * 60)
print("  Checkpoint Inspection")
print("=" * 60)
print(f"Latest Checkpoint Path : {LATEST_CHECKPOINT_PATH}")
print(f"Checkpoint Exists      : {LATEST_CHECKPOINT_PATH.exists()}")

if LATEST_CHECKPOINT_PATH.exists():
    chk = torch.load(LATEST_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    print(f"  Last Saved Epoch     : {chk.get('epoch', 'N/A')}")
    print(f"  Best Val F1 So Far   : {chk.get('best_metric_val', chk.get('best_f1', 'N/A')):.4f}")
    print(f"  History Entries      : {len(chk.get('history', []))}")
    print("\n[Action] Set RESUME_IF_AVAILABLE = True to resume, or False to restart.")
else:
    print("\nNo existing checkpoint found. Fresh training will begin from Epoch 1.")


## 6. End-to-End ResNet34-FCN Training Loop (with AMP & Drive Sync)


In [ ]:
# 6.1 Resilient Training Execution with Live Batch & Epoch Progress
import csv
import time
from tqdm.auto import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from src.losses import CombinedBCEDiceLoss
from src.metrics import MetricTracker

# 1. Optimizer, Criterion, Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = CombinedBCEDiceLoss(bce_weight=0.5, dice_weight=0.5).to(device)
scaler = torch.amp.GradScaler("cuda")

# 2. Schedulers (Linear Warmup + Cosine Annealing)
if WARMUP_EPOCHS > 0 and TOTAL_EPOCHS > WARMUP_EPOCHS:
    warmup_sched = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
    cosine_sched = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])
else:
    scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)

# 3. State tracking
start_epoch = 1
best_val_f1 = -1.0
best_epoch = 0
history = []
csv_log_path = EXP_LOGS / "training_history.csv"

# Initialize CSV log if not present
if not csv_log_path.exists():
    with open(csv_log_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_loss", "train_iou", "train_f1", "train_precision", "train_recall",
            "val_loss", "val_iou", "val_f1", "val_precision", "val_recall", "val_accuracy",
            "learning_rate", "epoch_time"
        ])

# 4. Resume state if requested and available
if RESUME_IF_AVAILABLE and LATEST_CHECKPOINT_PATH.exists():
    print(f"\nResuming training from {LATEST_CHECKPOINT_PATH} ...")
    checkpoint = torch.load(LATEST_CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    if "optimizer_state_dict" in checkpoint and checkpoint["optimizer_state_dict"] is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    if "scheduler_state_dict" in checkpoint and checkpoint["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        
    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_val_f1 = float(checkpoint.get("best_metric_val", checkpoint.get("best_f1", -1.0)))
    best_epoch = int(checkpoint.get("best_epoch", 0))
    history = list(checkpoint.get("history", []))
    print(f"✓ Resumed: Next Epoch = {start_epoch}, Best F1 So Far = {best_val_f1:.4f} (Epoch {best_epoch})\n")

# 5. Main Training Cycle
print("=" * 70)
print(f"  Starting ResNet34-FCN Training: Epochs {start_epoch} -> {TOTAL_EPOCHS}")
print("=" * 70)

for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    t_epoch_start = time.time()
    
    # ------------------ TRAINING PASS ------------------
    model.train()
    train_tracker = MetricTracker(threshold=0.5)
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{TOTAL_EPOCHS:02d} [Train]", leave=False)
    
    for images, masks in train_pbar:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        masks = masks.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, masks)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        batch_m = train_tracker.update(logits.detach(), masks, loss=loss.item())
        train_pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            f1=f"{batch_m['f1']:.3f}",
            iou=f"{batch_m['iou']:.3f}"
        )
        
    train_metrics = train_tracker.compute()
    
    # ------------------ VALIDATION PASS ------------------
    model.eval()
    val_tracker = MetricTracker(threshold=0.5)
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{TOTAL_EPOCHS:02d} [Val]  ", leave=False)
    
    with torch.no_grad():
        for images, masks in val_pbar:
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            masks = masks.to(device, non_blocking=True)
            
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                val_loss = criterion(logits, masks)
                
            batch_vm = val_tracker.update(logits, masks, loss=val_loss.item())
            val_pbar.set_postfix(
                val_loss=f"{val_loss.item():.4f}",
                val_f1=f"{batch_vm['f1']:.3f}",
                val_iou=f"{batch_vm['iou']:.3f}"
            )
            
    val_metrics = val_tracker.compute()
    
    # Step scheduler
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()
    
    epoch_duration = time.time() - t_epoch_start
    is_best = val_metrics["f1"] > best_val_f1
    if is_best:
        best_val_f1 = val_metrics["f1"]
        best_epoch = epoch
        
    # Record history
    record = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_iou": train_metrics["iou"],
        "train_f1": train_metrics["f1"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "val_loss": val_metrics["loss"],
        "val_iou": val_metrics["iou"],
        "val_f1": val_metrics["f1"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_accuracy": val_metrics["accuracy"],
        "learning_rate": current_lr,
        "epoch_time": round(epoch_duration, 2),
    }
    history.append(record)
    
    # ------------------ ATOMIC CHECKPOINT SAVING ------------------
    chk_payload = {
        "epoch": epoch,
        "model_name": "resnet34_fcn",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_metric_val": best_val_f1,
        "best_f1": best_val_f1,
        "best_epoch": best_epoch,
        "history": history,
        "batch_size": BATCH_SIZE,
        "amp": True
    }
    
    # Save latest atomically
    tmp_latest = EXP_CHECKPOINTS / "latest_model.pth.tmp"
    torch.save(chk_payload, tmp_latest)
    tmp_latest.replace(LATEST_CHECKPOINT_PATH)
    
    # Save best atomically
    if is_best:
        tmp_best = EXP_CHECKPOINTS / "best_model.pth.tmp"
        torch.save(chk_payload, tmp_best)
        tmp_best.replace(BEST_CHECKPOINT_PATH)
        
    # Append to CSV log
    with open(csv_log_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            record["epoch"],
            f"{record['train_loss']:.4f}",
            f"{record['train_iou']:.4f}",
            f"{record['train_f1']:.4f}",
            f"{record['train_precision']:.4f}",
            f"{record['train_recall']:.4f}",
            f"{record['val_loss']:.4f}",
            f"{record['val_iou']:.4f}",
            f"{record['val_f1']:.4f}",
            f"{record['val_precision']:.4f}",
            f"{record['val_recall']:.4f}",
            f"{record['val_accuracy']:.4f}",
            f"{record['learning_rate']:.2e}",
            record["epoch_time"]
        ])
        
    # Epoch summary print
    print(
        f"Epoch [{epoch:02d}/{TOTAL_EPOCHS:02d}] ({epoch_duration:.1f}s) | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val IoU: {val_metrics['iou']:.4f} | "
        f"LR: {current_lr:.1e} "
        f"{'[★ BEST]' if is_best else ''}"
    )

print("\n" + "=" * 70)
print(f"  Training Completed! Best Epoch = {best_epoch} with Val F1 = {best_val_f1:.4f}")
print("=" * 70)


## 7. Training History Visualization


In [ ]:
# 7.1 Plot Loss, F1 Score, and IoU Curves
import pandas as pd
import matplotlib.pyplot as plt

if csv_log_path.exists():
    df_history = pd.read_csv(csv_log_path)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Loss
    axes[0].plot(df_history["epoch"], df_history["train_loss"], label="Train Loss", marker="o", color="#1f77b4")
    axes[0].plot(df_history["epoch"], df_history["val_loss"], label="Val Loss", marker="s", color="#ff7f0e")
    axes[0].set_title("Combined BCE + Dice Loss", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle="--", alpha=0.6)
    axes[0].legend()
    
    # 2. F1 / Dice Score
    axes[1].plot(df_history["epoch"], df_history["train_f1"], label="Train F1", marker="o", color="#1f77b4")
    axes[1].plot(df_history["epoch"], df_history["val_f1"], label="Val F1", marker="s", color="#2ca02c")
    axes[1].set_title("Dice Coefficient / F1-Score", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("F1 Score")
    axes[1].grid(True, linestyle="--", alpha=0.6)
    axes[1].legend()
    
    # 3. IoU
    axes[2].plot(df_history["epoch"], df_history["train_iou"], label="Train IoU", marker="o", color="#1f77b4")
    axes[2].plot(df_history["epoch"], df_history["val_iou"], label="Val IoU", marker="s", color="#d62728")
    axes[2].set_title("Intersection over Union (IoU)", fontsize=12, fontweight="bold")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("IoU")
    axes[2].grid(True, linestyle="--", alpha=0.6)
    axes[2].legend()
    
    plt.tight_layout()
    curve_path = EXP_PLOTS / "resnet34_fcn_training_curves.png"
    plt.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Training curves saved to: {curve_path}")
else:
    print("No training history CSV found yet.")


## 8. Full Validation Benchmark Evaluation
Execute `scripts/evaluate.py` across all **2,367 validation samples** ($620,494,848$ total pixels) using the best saved ResNet34-FCN checkpoint.


In [ ]:
# 8.1 Evaluate Best ResNet34-FCN Checkpoint
import json
import subprocess

print("Running scripts/evaluate.py on best ResNet34-FCN checkpoint...")

eval_cmd = [
    sys.executable, "scripts/evaluate.py",
    "--checkpoint", str(BEST_CHECKPOINT_PATH),
    "--model", "resnet34_fcn",
    "--val-dir", str(LOCAL_VAL_DIR),
    "--output-dir", str(EXP_EVALUATION),
    "--batch-size", "8",
    "--device", "cuda" if torch.cuda.is_available() else "cpu"
]

!{" ".join(eval_cmd)}

# Display generated evaluation summary
fcn_eval_json = EXP_EVALUATION / "eval_metrics.json"
fcn_eval_txt = EXP_EVALUATION / "eval_summary.txt"

if fcn_eval_json.exists():
    with open(fcn_eval_json) as f:
        eval_data = json.load(f)
    
    m = eval_data["metrics"]
    cm = eval_data["confusion_matrix"]
    print("\n" + "=" * 60)
    print("  ResNet34-FCN Full Validation Results (N=2,367)")
    print("=" * 60)
    print(f"Dice / F1-Score : {m['f1'] * 100:.2f}% ({m['f1']:.6f})")
    print(f"Mean IoU        : {m['iou'] * 100:.2f}% ({m['iou']:.6f})")
    print(f"Precision       : {m['precision'] * 100:.2f}% ({m['precision']:.6f})")
    print(f"Recall          : {m['recall'] * 100:.2f}% ({m['recall']:.6f})")
    print(f"Pixel Accuracy  : {m['accuracy'] * 100:.2f}% ({m['accuracy']:.6f})")
    print("-" * 60)
    print(f"True Positives  : {cm['TP']:,}")
    print(f"False Positives : {cm['FP']:,}")
    print(f"False Negatives : {cm['FN']:,}")
    print(f"True Negatives  : {cm['TN']:,}")
    print("=" * 60)


## 9. DeepLabV3+ vs. ResNet34-FCN Comparison

Compare the **DeepLabV3+ advanced model** (ResNet-50 + ASPP) against the **ResNet34-FCN baseline** on the identical validation split ($N=2,367$).

> **Important Note on Academic Integrity**:
> All reported metrics are evaluated on the held-out validation split of the GLID dataset. They serve to quantify architectural improvements (ASPP multi-scale context and low-level boundary fusion) under identical data preprocessing, loss functions, and evaluation thresholds.


In [ ]:
# 9.1 Generate Side-by-Side Model Comparison Table
from pathlib import Path
import json

# Locate DeepLabV3+ Checkpoint and Evaluation JSON
dlv3_candidates = [
    DRIVE_PROJECT_ROOT / "training" / "deeplabv3plus_fast" / "checkpoints" / "best_amp_model.pth",
    DRIVE_PROJECT_ROOT / "checkpoints" / "best_amp_model.pth",
    REPO_DIR / "checkpoints" / "best_amp_model.pth"
]
dlv3_eval_candidates = [
    DRIVE_PROJECT_ROOT / "training" / "deeplabv3plus_fast" / "results" / "evaluation" / "eval_metrics.json",
    REPO_DIR / "results" / "evaluation" / "eval_metrics.json"
]

dlv3_chk = next((p for p in dlv3_candidates if p.is_file()), None)
dlv3_eval = next((p for p in dlv3_eval_candidates if p.is_file()), None)

print(f"DeepLabV3+ Checkpoint : {dlv3_chk}")
print(f"DeepLabV3+ Eval JSON  : {dlv3_eval}")
print(f"ResNet34-FCN Checkpoint: {BEST_CHECKPOINT_PATH}")
print(f"ResNet34-FCN Eval JSON : {fcn_eval_json}")

# Output comparison directory
COMPARE_OUT_DIR = EXP_DIR / "comparison"
COMPARE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Run compare_models.py script if checkpoints exist
if BEST_CHECKPOINT_PATH.exists():
    compare_cmd = [
        sys.executable, "scripts/compare_models.py",
        "--fcn-checkpoint", str(BEST_CHECKPOINT_PATH),
        "--fcn-eval", str(fcn_eval_json),
        "--output-dir", str(COMPARE_OUT_DIR)
    ]
    if dlv3_chk:
        compare_cmd.extend(["--dlv3-checkpoint", str(dlv3_chk)])
    if dlv3_eval:
        compare_cmd.extend(["--dlv3-eval", str(dlv3_eval)])
        
    !{" ".join(compare_cmd)}
    
    table_md = COMPARE_OUT_DIR / "comparison_table.md"
    if table_md.exists():
        print("\n" + "=" * 60)
        print("  ARCHITECTURAL BENCHMARK COMPARISON TABLE")
        print("=" * 60)
        with open(table_md) as f:
            print(f.read())
else:
    print("ResNet34-FCN evaluation not found yet. Run training and evaluation cells first.")


## 10. Qualitative Prediction Visualizations
Render 4-panel visual comparison figures (Satellite RGB, Ground Truth, Predicted Mask, Probability Heatmap) for sample validation scenes.


In [ ]:
# 10.1 Generate 4-panel visual predictions on validation samples
import matplotlib.pyplot as plt
import numpy as np

model.eval()
vis_loader = DataLoader(val_ds, batch_size=6, shuffle=False)
imgs, msks = next(iter(vis_loader))

with torch.no_grad():
    x = imgs.to(device, memory_format=torch.channels_last)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(x)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

imgs_np = imgs.permute(0, 2, 3, 1).cpu().numpy()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
imgs_np = np.clip(imgs_np * std + mean, 0.0, 1.0)

gt_np = msks.squeeze(1).cpu().numpy()
pred_np = preds.squeeze(1).cpu().numpy()
prob_np = probs.squeeze(1).cpu().numpy()

fig, axes = plt.subplots(4, 6, figsize=(18, 12))

for i in range(6):
    # RGB
    axes[0, i].imshow(imgs_np[i])
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_ylabel("Satellite RGB", fontsize=10, fontweight="bold")
        
    # Ground Truth
    axes[1, i].imshow(gt_np[i], cmap="Blues_r", vmin=0, vmax=1)
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_ylabel("Ground Truth", fontsize=10, fontweight="bold")
        
    # Predicted Mask
    axes[2, i].imshow(pred_np[i], cmap="Blues_r", vmin=0, vmax=1)
    axes[2, i].axis("off")
    if i == 0:
        axes[2, i].set_ylabel("ResNet34-FCN Pred", fontsize=10, fontweight="bold")
        
    # Probability Heatmap
    im = axes[3, i].imshow(prob_np[i], cmap="viridis", vmin=0, vmax=1)
    axes[3, i].axis("off")
    if i == 0:
        axes[3, i].set_ylabel("Lake Probability", fontsize=10, fontweight="bold")

plt.suptitle("ResNet34-FCN Validation Predictions", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
vis_save_path = EXP_PREDICTIONS / "sample_validation_predictions.png"
plt.savefig(vis_save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Visual predictions saved to: {vis_save_path}")
